# Fabric Cost Ingestion via API (notebook-only pattern)

This notebook is a lightweight alternative to the export-based pipeline used by the
[**Fabric Cost Analysis (FCA)**](https://github.com/microsoft/fabric-toolbox/tree/main/monitoring/fabric-cost-analysis)
solution accelerator.

Instead of configuring an Azure **Cost Management export** to a Data Lake Gen2 account and
shortcutting it into OneLake, this notebook calls the **Azure Cost Management Query API**
directly from a Fabric notebook, using the identity of the account (or service principal)
running the notebook, and lands the result straight into a Lakehouse Delta table.

Use this pattern when:
- You want cost data without provisioning a separate ADLS Gen2 account / scheduled export
- You need cost figures more frequently than the export's daily cadence allows
- You are prototyping and want a single self-contained notebook

Use the full FCA pipeline pattern instead when:
- You need the full FOCUS 1.0 schema (this API returns a reduced, Cost-Management-specific
  shape, not true FOCUS)
- You manage many subscriptions/billing accounts and want the export's built-in monthly chunking
- You want the full FCA semantic model + report out of the box

**Required permissions**
- The identity running this notebook needs **Cost Management Reader** (or **Billing Reader**)
  on every scope (subscription / resource group / billing account) you query.
- To enrich with Fabric capacity metadata (Step 4) the identity needs the **Fabric Administrator**
  role, and [tenant admin APIs](https://learn.microsoft.com/en-us/fabric/admin/tenant-settings-index#developer-settings)
  must be enabled for the tenant.

## Step 0 – Parameters

In [ ]:
from notebookutils import mssparkutils
import requests, json
from datetime import date, timedelta
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql import Row

# --- Cost Management scope(s) to query ---
# Subscription scope example: "/subscriptions/<subscription-id>"
# Resource group scope:       "/subscriptions/<subscription-id>/resourceGroups/<rg-name>"
# Billing account scope (EA/MCA): "/providers/Microsoft.Billing/billingAccounts/<id>"
scopes = [
    "/subscriptions/00000000-0000-0000-0000-000000000000"
]

# Reporting window (inclusive)
start_date = (date.today().replace(day=1) - timedelta(days=1)).replace(day=1)  # first day of previous month
end_date = date.today()

# Lakehouse target table (must run in a notebook attached to a Lakehouse)
target_table = "cost_fabric_api"

api_version = "2023-11-01"
management_endpoint = "https://management.azure.com"

# --- Service Principal auth (recommended) ---
# Leave spn_client_id empty to fall back to your own delegated identity via
# notebookutils.credentials.getToken -- that path goes through Fabric's own
# token-broker service, which has shown intermittent 500 "TM" internal errors.
# Setting spn_client_id switches to a direct OAuth2 client-credentials call
# that bypasses that broker entirely -- see the README's "Service Principal
# setup" section for the one-time az CLI steps.
tenant_id = "<aad-tenant-id>"
spn_client_id = ""  # set this to opt into Service Principal auth
key_vault_url = "https://<your-keyvault-name>.vault.azure.net/"
key_vault_secret_name = "<secret-name-holding-the-spn-client-secret>"
# Emergency-only fallback if Key Vault access isn't wired up yet.
# NEVER commit a real secret here -- this is for local testing only.
spn_client_secret_override = ""

## Step 1 – Authenticate

Two options, controlled by whether `spn_client_id` (set above) is filled in:

- **Service Principal (recommended)** — a direct OAuth2 client-credentials call to Azure AD.
  This does not touch Fabric's own token-broker service at all, so it isn't exposed to that
  broker's reliability issues. Needs a one-time setup (see the README's
  "Service Principal setup" section) and the SPN granted **Cost Management Reader** on your
  scope(s).
- **Your own delegated identity (default/fallback)** — `notebookutils.credentials.getToken`,
  which runs as whoever executes the notebook, no secrets involved. Simpler to start with, but
  routes through Fabric's token-broker (`Trident TokenLibrary`), which has shown intermittent
  `500 INTERNAL_ERROR` / `Source: TM` failures for the ARM audience — see the README's
  "Troubleshooting" section if you hit that.

In [ ]:
import time

def get_client_secret():
    if spn_client_secret_override:
        print("WARN: using spn_client_secret_override -- move this to Key Vault before scheduling/sharing this notebook.")
        return spn_client_secret_override
    return notebookutils.credentials.getSecret(key_vault_url, key_vault_secret_name)

def get_arm_token_via_spn(tenant_id, client_id, client_secret):
    token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    resp = requests.post(token_url, data={
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": "https://management.azure.com/.default"
    })
    resp.raise_for_status()
    return resp.json()["access_token"]

def get_arm_token_via_delegated_identity(max_attempts=3, base_delay_seconds=5):
    """
    notebookutils.credentials.getToken occasionally fails with a transient
    500 INTERNAL_ERROR from Fabric's token broker (Token Management / "TM"
    service) rather than an actual auth/permission problem. Retry with
    backoff before surfacing the error, and restart the notebook session if
    it still fails after these retries -- that clears a stuck broker session,
    which is the most common fix. If it keeps recurring across fresh
    sessions, switch to Service Principal auth (set spn_client_id above)
    instead of continuing to retry this path.
    """
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return notebookutils.credentials.getToken("https://management.azure.com/")
        except Exception as e:
            last_error = e
            print(f"WARN: getToken attempt {attempt}/{max_attempts} failed: {e}")
            if attempt < max_attempts:
                time.sleep(base_delay_seconds * attempt)
    raise RuntimeError(
        "Failed to acquire an ARM token after retries via the delegated identity path. "
        "Restart the notebook session (Stop session, then run again) and retry once more. "
        "If it keeps recurring, switch to Service Principal auth by setting spn_client_id "
        "in the Parameters cell -- see the README's 'Service Principal setup' section."
    ) from last_error

def get_arm_token():
    if spn_client_id:
        client_secret = get_client_secret()
        return get_arm_token_via_spn(tenant_id, spn_client_id, client_secret)
    return get_arm_token_via_delegated_identity()

arm_token = get_arm_token()
headers = {
    "Authorization": f"Bearer {arm_token}",
    "Content-Type": "application/json"
}

## Step 2 – Query cost data (Fabric-only, FOCUS-ish shape)

We call `POST {scope}/providers/Microsoft.CostManagement/query`, filtered to
`ServiceName eq 'Microsoft Fabric'`, grouped by resource/meter, and follow `nextLink`
until fully paged (Cost Management returns up to 2,000 rows per page).

In [ ]:
def build_query_body(start_date, end_date):
    return {
        "type": "ActualCost",
        "timeframe": "Custom",
        "timePeriod": {
            "from": start_date.strftime("%Y-%m-%dT00:00:00Z"),
            "to": end_date.strftime("%Y-%m-%dT00:00:00Z")
        },
        "dataset": {
            "granularity": "Daily",
            "aggregation": {
                "totalCost": {"name": "Cost", "function": "Sum"},
                "totalCostUSD": {"name": "CostUSD", "function": "Sum"}
            },
            "grouping": [
                {"type": "Dimension", "name": "ResourceId"},
                {"type": "Dimension", "name": "ResourceGroupName"},
                {"type": "Dimension", "name": "MeterCategory"},
                {"type": "Dimension", "name": "MeterSubCategory"},
                {"type": "Dimension", "name": "ResourceLocation"}
            ],
            "filter": {
                "dimensions": {
                    "name": "ServiceName",
                    "operator": "In",
                    "values": ["Microsoft Fabric"]
                }
            }
        }
    }

def query_cost_management(scope, headers, body):
    url = f"{management_endpoint}{scope}/providers/Microsoft.CostManagement/query?api-version={api_version}"
    columns, rows = None, []
    next_body = dict(body)
    while True:
        resp = requests.post(url, headers=headers, data=json.dumps(next_body))
        if resp.status_code == 429:
            # basic throttling backoff, honor Retry-After when present
            import time
            time.sleep(int(resp.headers.get("Retry-After", "10")))
            continue
        resp.raise_for_status()
        payload = resp.json()
        properties = payload["properties"]
        if columns is None:
            columns = [c["name"] for c in properties["columns"]]
        rows.extend(properties["rows"])
        next_link = properties.get("nextLink")
        if not next_link:
            break
        url = next_link
        next_body = None  # nextLink already encodes the full request for GET-style paging
    return columns, rows

In [ ]:
all_rows = []
columns = None
body = build_query_body(start_date, end_date)

for scope in scopes:
    print(f"Querying cost for scope: {scope}")
    cols, rows = query_cost_management(scope, headers, body)
    columns = columns or cols
    for r in rows:
        all_rows.append(r + [scope])

print(f"Retrieved {len(all_rows)} rows across {len(scopes)} scope(s)")

## Step 3 – Land into a Lakehouse Delta table

In [ ]:
row_columns = columns + ["Scope"]
spark_rows = [Row(**dict(zip(row_columns, r))) for r in all_rows]
df = spark.createDataFrame(spark_rows) if spark_rows else None

if df is not None:
    df = df.withColumn("IngestedAt", current_timestamp())

    # Clean any previous snapshot for the same window before reloading (idempotent re-runs)
    if spark.catalog.tableExists(target_table):
        spark.sql(f"""
            DELETE FROM {target_table}
            WHERE UsageDate >= '{start_date.strftime('%Y-%m-%d')}'
              AND UsageDate <= '{end_date.strftime('%Y-%m-%d')}'
        """)
        df.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable(target_table)
    else:
        df.write.mode("overwrite").format("delta").saveAsTable(target_table)

    print(f"Wrote {df.count()} rows to Delta table '{target_table}'")
else:
    print("No rows returned for the given window/scope(s) — nothing written.")

## Step 4 – (Optional) Enrich with Fabric capacity metadata via the Fabric Admin API

The Fabric REST **Admin APIs** (audience `https://api.fabric.microsoft.com/`) let a Fabric
Administrator list every capacity, workspace and item in the tenant — useful for turning
the raw `ResourceId` from Cost Management into a friendly capacity/workspace name.
Requires the **Fabric Administrator** role and tenant-admin API access enabled.

In [ ]:
fabric_token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com/")
fabric_headers = {"Authorization": f"Bearer {fabric_token}"}

resp = requests.get("https://api.fabric.microsoft.com/v1/admin/capacities", headers=fabric_headers)
resp.raise_for_status()
capacities = resp.json().get("capacities", [])

if capacities:
    cap_df = spark.createDataFrame(
        [Row(CapacityId=c["id"], CapacityName=c["displayName"], Sku=c.get("sku"), Region=c.get("region"))
         for c in capacities]
    )
    cap_df.write.mode("overwrite").format("delta").saveAsTable("dim_fabric_capacities")
    print(f"Loaded {cap_df.count()} capacities into 'dim_fabric_capacities'")

## Next steps

- Schedule this notebook directly (Fabric notebook **Schedule**) or orchestrate it from a
  Data Pipeline **Notebook activity**, similar to how FCA's `Load FCA E2E` pipeline drives
  its notebooks — this keeps a single place to manage `FromMonth`/`ToMonth`-style
  parameters.
- Join `cost_fabric_api` to `dim_fabric_capacities` (and a standard calendar dimension) to
  build a star schema, then create a Direct Lake **semantic model** and **Power BI report**
  on top — you can reuse the layout ideas from FCA's `FCA_Core_SM` / `FCA_Core_Report`
  (Home, Summary, Capacity Usage, Cost Detail pages) even though the source table here is
  narrower than full FOCUS.
- If you outgrow the API's per-query limits (2,000 rows/page, request throttling, ~1 year
  lookback) or need the full FOCUS schema, migrate to FCA's native Cost Management
  **export + OneLake shortcut** pattern documented in
  [`Deploy.md`](../../Deploy.md) — the ingestion notebooks in `../../src` can then take over.